# End-to-end forecasting with skforecast-ai

## What is skforecast-ai?

[**skforecast**](https://skforecast.org) is a Python library for time series forecasting using scikit-learn compatible models, statistical methods, and foundation models. It gives you the building blocks (forecasters, backtesting, feature engineering, hyperparameter search) but leaves the many design decisions (which forecaster? which lags? how to validate?) up to you.

**skforecast-ai** is an assistant built **on top of skforecast** that helps automate, inspect, and explain forecasting workflows. It is organized around a single object, the `ForecastingAssistant`, with two complementary parts:

- **Deterministic engine (rule-based, reproducible).** It profiles the data, selects a forecaster and estimator, derives lags and preprocessing, runs backtesting, and produces the final forecast, together with the **exact standalone skforecast script** that generated the results. For the same inputs and configuration, this workflow is designed to be reproducible.

- **Optional LLM support.** The `ask()` method explains the objects and results you pass to it: profiles, plans, validation choices, backtesting outputs, forecasts, or general forecasting questions. In `ask()`, the LLM interprets existing information, it does not rerun the workflow or silently change the recommendation. Agentic features such as LLM-guided `refine_plan()` or `create_cv()` are separate, explicit steps where the LLM can suggest changes that are then implemented in deterministic code.

<p style="font-size:1.17em; font-weight:600; margin:1em 0; line-height:1.25;">Deterministic first, LLM where it helps</p>

The library is built around a practical split: forecasting decisions should be inspectable and reproducible, while natural-language explanations can be optional. The deterministic methods produce the configuration, results, and code. The LLM layer is useful when you want help understanding why a choice was made, how to read a backtest, or what a forecast implies.

That means you can run the full workflow without an LLM. If you do use one, it is mainly there to explain the artifacts produced by the workflow, or to support explicit agentic steps.

## Two ways to use skforecast-ai

`skforecast-ai` supports two ways of working with the same forecasting engine.

+ Use the **fast path** when you want a forecast or backtest result in a single call. The assistant profiles the data, builds the plan, runs the workflow, and returns the results together with the reproducible **skforecast** code.

+ Use the **step-by-step path** when you want to inspect or adjust the intermediate decisions. Create a profile, build a plan, optionally refine it, define a validation strategy, evaluate the model, and then generate the forecast.

A useful distinction is that forecasting and validation are separate branches. Once you have a `profile` and a `plan`, `forecast()` can produce future predictions directly and `backtest()` can evaluate the model on historical data.

You can call `ask()` in either workflow. It can explain a profile, a plan, a validation setup, a backtest result, a forecast result, or answer a general forecasting question. These calls are explanatory: they help interpret the current object or result, but they do not execute the workflow or silently modify the recommendation.

<div style="box-sizing:border-box; margin:16px 0; font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif; color:#24292f; max-width:100%;">
  <div style="box-sizing:border-box; display:flex; gap:20px; flex-wrap:wrap; align-items:stretch;">

<!-- Fast path -->
<div style="box-sizing:border-box; flex:1 1 260px; min-width:0; border:1px solid #d0d7de; border-radius:12px; overflow:hidden; display:flex; flex-direction:column;">
    <div style="box-sizing:border-box; background:#0969da; color:#ffffff; padding:12px 16px; font-size:15px; font-weight:700;">Fast path &mdash; one call</div>
    <div style="box-sizing:border-box; padding:16px; background:#f6f8fa; flex:1;">
    <p style="margin:0 0 12px 0; font-size:13px;">Profiling, planning and execution happen internally.</p>
    <div style="box-sizing:border-box; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px 12px; text-align:center; font-weight:600;">data</div>
    <div style="text-align:center; color:#57606a; font-size:18px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; display:flex; gap:12px; flex-wrap:wrap;">
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Forecast</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">forecast()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or forecast_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">predictions + code</div>
        </div>
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Backtesting (validation)</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">create_cv()<br><span style="font-weight:400; font-size:12px; color:#57606a;">Deterministic, Agentic mode</span><br><span style="font-weight:400; font-size:12px; color:#57606a;">or pass a skforecast TimeSeriesFold object</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="box-sizing:border-box; background:#dbeafe; border:1px solid #0969da; border-radius:8px; padding:8px; text-align:center; font-weight:700;">backtest()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or backtest_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">metrics + predictions + code</div>
        </div>
    </div>
    </div>
</div>

<!-- Step-by-step path -->
<div style="box-sizing:border-box; flex:1.6 1 340px; min-width:0; border:1px solid #d0d7de; border-radius:12px; overflow:hidden; display:flex; flex-direction:column;">
    <div style="box-sizing:border-box; background:#1a7f37; color:#ffffff; padding:12px 16px; font-size:15px; font-weight:700;">Step-by-step path &mdash; full control</div>
    <div style="box-sizing:border-box; padding:16px; background:#f6f8fa; flex:1;">
    <p style="margin:0 0 12px 0; font-size:13px;">Build a <code>profile</code> and a <code>plan</code> from your data, then branch into forecasting and backtesting.</p>
    <div style="box-sizing:border-box; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:8px 12px; text-align:center; font-weight:600;">data</div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px 12px; text-align:center; font-weight:700;">profile()</div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px 12px; text-align:center; font-weight:700;">plan()<br><span style="font-weight:400; font-size:12px; color:#57606a;">refine_plan(), optional (Deterministic or Agentic mode)</span></div>
    <div style="text-align:center; color:#57606a; font-size:16px; line-height:1.4;">&#8595;</div>
    <div style="box-sizing:border-box; display:flex; gap:12px; flex-wrap:wrap;">
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Forecast</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">forecast()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or forecast_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">predictions + code</div>
        </div>
        <div style="box-sizing:border-box; flex:1 1 150px; min-width:0; background:#ffffff; border:1px solid #d0d7de; border-radius:8px; padding:10px;">
        <div style="font-size:11px; color:#57606a; text-transform:uppercase; letter-spacing:.5px; text-align:center; margin-bottom:6px;">Backtesting (validation)</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">create_cv()<br><span style="font-weight:400; font-size:12px; color:#57606a;">Deterministic, Agentic mode</span><br><span style="font-weight:400; font-size:12px; color:#57606a;">or pass a skforecast TimeSeriesFold object</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="box-sizing:border-box; background:#dcfce7; border:1px solid #1a7f37; border-radius:8px; padding:8px; text-align:center; font-weight:700;">backtest()<br><span style="font-weight:400; font-size:12px; color:#57606a;">or backtest_code()</span></div>
        <div style="text-align:center; color:#57606a; font-size:15px; line-height:1.4;">&#8595;</div>
        <div style="text-align:center; font-size:12px; color:#24292f;">metrics + predictions + code</div>
        </div>
    </div>
    </div>
</div>

  </div>

  <!-- ask() banner -->
  <div style="box-sizing:border-box; margin-top:16px; border:1px solid #8250df; border-radius:12px; overflow:hidden;">
    <div style="box-sizing:border-box; background:#8250df; color:#ffffff; padding:10px 16px; font-size:15px; font-weight:700;">ask() &mdash; available at any moment, in any workflow</div>
    <div style="box-sizing:border-box; padding:12px 16px; background:#faf5ff; font-size:13px;">Call <code>ask()</code> before, during or after either path. It can take a <code>profile</code>, a <code>plan</code>, a <code>forecast_result</code>, a <code>backtest_result</code>, or nothing at all (pure Q&amp;A). It only explains, it never changes a recommendation.</div>
  </div>
</div>

<div role="note"
     style="background: rgba(0,184,212,.08); border-left: 6px solid #00b8d4; border-radius: 6px; padding: 10px 12px; margin: 1em 0;">
  <p style="display: flex; align-items: center; font-size: 1rem; color: #00b8d4; margin: 0 0 6px 0; font-weight: 600;">
    <span style="margin-right: 6px; font-size: 1.125em;">✏️</span>
    <strong style="font-size: 18px;">Note</strong>
  </p>

  <p>
    Both paths use the same deterministic workflow and return reproducible <strong>skforecast</strong> code. The step-by-step path additionally exposes intermediate objects, such as <code>profile</code>, <code>plan</code>, and <code>cv</code>, so you can inspect them or adjust them before running the next step.
  </p>
  
</div>

## Setup

In [1]:
# Data processing
# ==============================================================================
import os
import numpy as np
import pandas as pd
from skforecast.datasets import fetch_dataset

# Plots
# ==============================================================================
from skforecast.plot import set_dark_theme
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as poff
pio.templates.default = "seaborn"
poff.init_notebook_mode(connected=True)
plt.style.use('seaborn-v0_8-darkgrid')

# skforecast and skforecast-ai
# ==============================================================================
import skforecast
import skforecast_ai
from skforecast_ai import ForecastingAssistant
from skforecast.model_selection import TimeSeriesFold

color = '\033[1m\033[38;5;208m' 
print(f"{color}Version skforecast_ai: {skforecast_ai.__version__}")
print(f"{color}Version skforecast: {skforecast.__version__}")

Version skforecast_ai: 0.1.0
Version skforecast: 0.23.0


We use two `ForecastingAssistant` instances in this tutorial:

- `assistant`: runs the deterministic workflow. We use it for profiling, planning, backtesting, and forecasting.

- `assistant_llm`: has an LLM configured. We use it for the cells that call `ask()` and for the optional LLM-guided `refine_plan()` and `create_cv()` examples.

The LLM is configured with a string in the form `'provider:model_name'`, for example `'openai:gpt-5.5'`, `'google:gemini-3-flash-preview'`, `'anthropic:claude-sonnet-5'`, or `'ollama:qwen3:8b'`. For hosted providers, the corresponding API key must be available through the environment or passed explicitly when creating the assistant. In this tutorial, `send_data_to_llm=False`, so the LLM receives metadata and summary statistics, but not the raw time series values.

In [2]:
# Deterministic assistant (no LLM required)
# ==============================================================================
assistant = ForecastingAssistant()

# LLM-enabled assistant (used only for ask())
# ==============================================================================
LLM_MODEL = "google:gemini-3-flash-preview"
api_key = os.getenv("GOOGLE_API_KEY")

assistant_llm = ForecastingAssistant(
    llm=LLM_MODEL, api_key=api_key, send_data_to_llm=False
)

## Data

The data in this document represent the hourly usage of the bike share system in the city of Washington, D.C. during the years 2011 and 2012. In addition to the number of users per hour, information about weather conditions and holidays is available. The original data was obtained from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset) and has been previously cleaned ([code](https://github.com/JoaquinAmatRodrigo/Estadistica-machine-learning-python/blob/master/code/prepare_bike_sharing_dataset.ipynb)) applying the following modifications:

+ Renamed columns with more descriptive names.

+ Renamed categories of the weather variables. The category of `heavy_rain` has been combined with that of `rain`.

+ Denormalized temperature, humidity and wind variables.

+ `date_time` variable created and set as index.

+ Imputed missing values by forward filling.

In [3]:
# Downloading data
# ==============================================================================
data = fetch_dataset('bike_sharing', raw=True)
data = data[['date_time', 'users', 'holiday', 'weather', 'temp']]
data['date_time'] = pd.to_datetime(data['date_time'], format='%Y-%m-%d %H:%M:%S')
data = data.set_index('date_time')
data = data.asfreq('h')
data = data.sort_index()
data.head()

╭───────────────────────────────── bike_sharing ──────────────────────────────────╮
│ Description:                                                                    │
│ Hourly usage of the bike share system in the city of Washington D.C. during the │
│ years 2011 and 2012. In addition to the number of users per hour, information   │
│ about weather conditions and holidays is available.                             │
│                                                                                 │
│ Source:                                                                         │
│ Fanaee-T,Hadi. (2013). Bike Sharing Dataset. UCI Machine Learning Repository.   │
│ https://doi.org/10.24432/C5W894.                                                │
│                                                                                 │
│ URL:                                                                            │
│ https://raw.githubusercontent.com/skforecast/skforecast-                        │
│ datasets/main/data/bike_sharing_dataset_clean.csv                               │
│                                                                                 │
│ Shape: 17544 rows x 12 columns                                                  │
╰─────────────────────────────────────────────────────────────────────────────────╯

,users,holiday,weather,temp
date_time,,,,
2011-01-01 00:00:00,16.0,0.0,clear,9.84
2011-01-01 01:00:00,40.0,0.0,clear,9.02
2011-01-01 02:00:00,32.0,0.0,clear,9.02
2011-01-01 03:00:00,13.0,0.0,clear,9.84
2011-01-01 04:00:00,1.0,0.0,clear,9.84


<div role="note"
     style="background: rgba(0,184,212,.08); border-left: 6px solid #00b8d4; border-radius: 6px; padding: 10px 12px; margin: 1em 0;">
  <p style="display: flex; align-items: center; font-size: 1rem; color: #00b8d4; margin: 0 0 6px 0; font-weight: 600;">
    <span style="margin-right: 6px; font-size: 1.125em;">✏️</span>
    <strong style="font-size: 18px;">Note</strong>
  </p>

  <p>
    Skforecast-ai is ready to preprocess the data, but it is recommended that users apply their own preprocessing steps before using the assistant. This ensures the data is in the desired format and any necessary transformations have been applied before proceeding with the forecasting workflow.
  </p>
  
</div>

In [4]:
# Interactive plot of time series
# ==============================================================================
fig = go.Figure()
fig.add_trace(go.Scatter(x=data.index, y=data['users'], mode='lines', name='Train'))
fig.update_layout(
    title  = 'Number of users',
    xaxis_title="Time",
    yaxis_title="Users",
    legend_title="Partition:",
    width=800,
    height=400,
    margin=dict(l=20, r=20, t=35, b=20),
    legend=dict(orientation="h", yanchor="top", y=1, xanchor="left", x=0.001)
)
fig.show()

For a deeper walkthrough of the exploratory analysis behind this dataset, see the skforecast example: [Forecasting time series with skforecast, XGBoost, LightGBM and CatBoost](https://cienciadedatos.net/documentos/py39-forecasting-time-series-with-skforecast-xgboost-lightgbm-catboost#Data_exploration).

## Fast path

In [5]:
# Hold out the last 36 hours as the forecast horizon
# ==============================================================================
result_fc = assistant.forecast(
    data        = data,
    target      = 'users',
    date_column = "date_time",
    steps       = 36,
    interval    = [0.1, 0.9],
    test_size   = 36
)

display(result_fc.metrics)
display(result_fc.predictions.head())

,series,MAE,MSE,MASE,MAPE
0,users,32.013374,2758.8615,0.497185,0.415688


,pred,lower_bound,upper_bound
2012-12-30 12:00:00,146.512223,111.097550,181.185404
2012-12-30 13:00:00,138.768717,96.950185,178.804105
2012-12-30 14:00:00,145.133323,92.701496,185.332563
2012-12-30 15:00:00,140.891323,89.525337,185.272888
2012-12-30 16:00:00,139.538959,86.994814,180.904345


In [6]:
result_fc.show_code()

╭──────────────────────────────────────────────── Generated code ─────────────────────────────────────────────────╮
│ import pandas as pd                                                                                             │
│ from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error             │
│ from skforecast.metrics import mean_absolute_scaled_error                                                       │
│ from lightgbm import LGBMRegressor                                                                              │
│ from skforecast.preprocessing import RollingFeatures, CalendarFeatures                                          │
│ from skforecast.recursive import ForecasterRecursive                                                            │
│                                                                                                                 │
│ # Load data                                                                                                     │
│ data = pd.read_csv('data.csv', index_col=0, parse_dates=True)                                                   │
│                                                                                                                 │
│ data = data.asfreq('h')                                                                                         │
│ data = data.sort_index()                                                                                        │
│                                                                                                                 │
│ # Train/test split                                                                                              │
│ end_train = '2012-12-30 11:00:00'  # last training date, adjust to change the split point                       │
│ data_train = data.loc[:end_train]                                                                               │
│ data_test  = data.loc[data.index > end_train]                                                                   │
│ exog_features = ['holiday', 'weather', 'temp']                                                                  │
│                                                                                                                 │
│ print(                                                                                                          │
│     f"Train dates : {data_train.index.min()} --- {data_train.index.max()}  (n={len(data_train)})"               │
│ )                                                                                                               │
│ print(                                                                                                          │
│     f"Test dates  : {data_test.index.min()} --- {data_test.index.max()}  (n={len(data_test)})"                  │
│ )                                                                                                               │
│                                                                                                                 │
│ window_features = RollingFeatures(                                                                              │
│     stats        = ['mean', 'std', 'mean', 'mean'],                                                             │
│     window_sizes = [3, 3, 24, 168],                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
│ calendar_features = CalendarFeatures(                                                  

In [7]:
# Hold out the last 36 hours as the forecast horizon
# ==============================================================================
end_train = '2012-08-31 23:59:00'
data_train = data.loc[: end_train, :]
data_test  = data.loc[end_train:, :]

result_fc = assistant.forecast(
    data        = data_train,
    target      = 'users',
    date_column = "date_time",
    steps       = 36,
    interval    = [0.1, 0.9],
    exog        = data_test[['holiday', 'weather', 'temp']]
)

display(result_fc.predictions.head())

,pred,lower_bound,upper_bound
2012-09-01 00:00:00,130.518181,104.995011,154.098369
2012-09-01 01:00:00,97.866265,71.684595,127.528292
2012-09-01 02:00:00,64.924807,38.821421,100.956224
2012-09-01 03:00:00,32.180436,12.852130,63.292462
2012-09-01 04:00:00,9.429745,2.916334,30.445517


In [8]:
result_fc.show_code()

╭──────────────────────────────────────────────── Generated code ─────────────────────────────────────────────────╮
│ import pandas as pd                                                                                             │
│ from lightgbm import LGBMRegressor                                                                              │
│ from skforecast.preprocessing import RollingFeatures, CalendarFeatures                                          │
│ from skforecast.recursive import ForecasterRecursive                                                            │
│                                                                                                                 │
│ # Load data                                                                                                     │
│ data = pd.read_csv('data.csv', index_col=0, parse_dates=True)                                                   │
│                                                                                                                 │
│ # Load future exogenous variables covering the forecast horizon                                                 │
│ exog_future = pd.read_csv('exog_future.csv', index_col=0, parse_dates=True)                                     │
│                                                                                                                 │
│ data = data.asfreq('h')                                                                                         │
│ data = data.sort_index()                                                                                        │
│                                                                                                                 │
│ # Prepare the future exogenous index                                                                            │
│ exog_future = exog_future.sort_index()                                                                          │
│ exog_future = exog_future.asfreq('h')                                                                           │
│                                                                                                                 │
│ exog_features = ['holiday', 'weather', 'temp']                                                                  │
│                                                                                                                 │
│ window_features = RollingFeatures(                                                                              │
│     stats        = ['mean', 'std', 'mean', 'mean'],                                                             │
│     window_sizes = [3, 3, 24, 168],                                                                             │
│ )                                                                                                               │
│                                                                                                                 │
│ calendar_features = CalendarFeatures(                                                                           │
│     features = ['hour', 'day_of_week', 'weekend'],                                                              │
│     encoding = None,                                                                                            │
│ )                                                                                                               │
│                                                                                                                 │
│ # Create forecaster                                                                                             │
│ forecaster = ForecasterRecursive(                                                                               │
│     estimator            = LGBMRegressor(random_state=123, verbose=-1),                                         │
│     lags                 = [1, 2, 3, 5, 8, 10, [38;2;

## Step-by-step path

### 3. Profile the data

`profile()` inspects the dataset and selects the recommended **forecaster** and **estimator** (plus alternative candidates). It also detects the frequency, seasonality, missing values, and the role of the exogenous variables. This is a deterministic step.

In [ ]:
profile = assistant.profile(
    data        = data,
    target      = target,
    date_column = "date_time",
)
profile

### `ask()` — Explain mode

We pass the pre-computed `profile` so no work is repeated: the LLM only explains the deterministic decision.

In [ ]:
answer = assistant_llm.ask(
    prompt  = (
        "Explain why this forecaster and estimator were recommended for my "
        "hourly bike-sharing demand data, and what the exogenous variables add."
    ),
    profile = profile,
    steps   = 36,
)
print(answer.explanation)

## 4. Build the plan

`plan()` turns the coarse decisions from the profile into a detailed, executable configuration: lags (via PACF), window features, preprocessing, NaN handling, exogenous usage, and prediction intervals. We forecast `steps=36` (a day and a half ahead) and request an 80% prediction interval.

In [ ]:
plan = assistant.plan(
    profile  = profile,
    steps    = 36,
    interval = [0.1, 0.9],
)
plan

### `ask()` — Explain mode (plan)

In [ ]:
answer = assistant_llm.ask(
    prompt  = (
        "Walk me through this plan. Why these lags and preprocessing steps, "
        "and how will the 80% prediction interval be produced?"
    ),
    profile = profile,
    plan    = plan,
)
print(answer.explanation)

## 5. Refine the plan

`refine_plan()` adjusts an existing plan. It works in two modes:

- **Deterministic** (`prompt=None`): apply explicit overrides; everything else is re-derived from the profile.
- **LLM-guided** (`prompt=...`): a specialized agent interprets domain knowledge and suggests `lags` / `window_features`.

Bike demand has a strong daily and weekly cycle, so we inject that domain knowledge. (Set `prompt=None` and pass explicit overrides such as `lags=[1, 2, 3, 24, 168]` if you prefer a fully deterministic refinement.)

In [ ]:
plan_refined = assistant_llm.refine_plan(
    profile = profile,
    plan    = plan,
    prompt  = (
        "Bike demand has strong daily (24h) and weekly (168h) seasonality, "
        "with sharp morning and evening commuting peaks on working days."
    ),
)
plan_refined

### `ask()` — Explain mode (refined plan)

In [ ]:
answer = assistant_llm.ask(
    prompt  = "What changed compared to the original plan, and why does it matter for this dataset?",
    profile = profile,
    plan    = plan_refined,
)
print(answer.explanation)

## 6. Define the validation strategy

`create_cv()` builds a `TimeSeriesFold` with smart defaults derived from the profile and plan, and returns a human-readable explanation. Here we fix the start of the backtest to the end of August 2012, so the model is evaluated on the last four months.

In [ ]:
cv, cv_explanation = assistant.create_cv(
    profile            = profile,
    plan               = plan_refined,
    initial_train_size = '2012-08-31 23:59:00',
    refit              = False,
)
print(cv_explanation)
cv

### `ask()` — Explain mode (cross-validation)

In [ ]:
answer = assistant_llm.ask(
    prompt  = (
        "Here is the chosen cross-validation setup:\n\n"
        f"{cv_explanation}\n\n"
        "Explain this configuration and whether disabling refit is reasonable here."
    ),
    profile = profile,
    plan    = plan_refined,
)
print(answer.explanation)

## 7. Evaluate with backtesting

`backtest()` runs time series cross-validation with the CV strategy and the pre-computed profile and plan, returning metrics, out-of-sample predictions, and the exact reproducible script.

In [ ]:
result_bt = assistant.backtest(
    data        = data,
    target      = target,
    date_column = "date_time",
    cv          = cv,
    profile     = profile,
    plan        = plan_refined,
)

display(result_bt.metrics)
display(result_bt.predictions.head())

In [ ]:
# Backtest predictions vs. actual values
# ==============================================================================
preds_bt = result_bt.predictions

fig, ax = plt.subplots(figsize=(11, 4))
data.loc[preds_bt.index, target].plot(ax=ax, label='actual', linewidth=1)
preds_bt['pred'].plot(ax=ax, label='backtest prediction', linewidth=1)
if {'lower_bound', 'upper_bound'}.issubset(preds_bt.columns):
    ax.fill_between(
        preds_bt.index,
        preds_bt['lower_bound'],
        preds_bt['upper_bound'],
        alpha=0.3,
        label='80% interval',
    )
ax.set_title('Backtesting: predictions vs. actual bike demand')
ax.set_ylabel('users')
ax.legend()
plt.show()

### `ask()` — Backtest mode

Passing `backtest_result` gives the LLM the metrics, predictions, and CV configuration so it can interpret the evaluation.

In [ ]:
answer = assistant_llm.ask(
    prompt          = (
        "Interpret these backtesting metrics. Is the model good enough to "
        "deploy, and where does it struggle?"
    ),
    backtest_result = result_bt,
)
print(answer.explanation)

## 8. Generate the forecast

Once the model is validated, we produce the actual forecast. The exogenous variables must cover the horizon, so we hold out the **last 36 hours** of the dataset: the model trains on everything before them, and we supply their known `exog` values as `exog_future`. Because we also have the true `users` for that window, we can visually check the forecast.

In [ ]:
# Hold out the last 36 hours as the forecast horizon
# ==============================================================================
steps = 36
data_train  = data.iloc[:-steps]
exog_future = data.iloc[-steps:][exog_cols]
y_future    = data.iloc[-steps:][target]

result_fc = assistant.forecast(
    data        = data_train,
    target      = target,
    date_column = "date_time",
    steps       = steps,
    interval    = [0.1, 0.9],
    exog_future = exog_future,
)

display(result_fc.metrics)
display(result_fc.predictions.head())

In [ ]:
# Forecast vs. actual values for the held-out horizon
# ==============================================================================
preds_fc = result_fc.predictions

fig, ax = plt.subplots(figsize=(11, 4))
data.loc['2012-12-20':, target].plot(ax=ax, label='actual', linewidth=1)
preds_fc['pred'].plot(ax=ax, label='forecast', linewidth=1.5)
if {'lower_bound', 'upper_bound'}.issubset(preds_fc.columns):
    ax.fill_between(
        preds_fc.index,
        preds_fc['lower_bound'],
        preds_fc['upper_bound'],
        alpha=0.3,
        label='80% interval',
    )
ax.set_title('36-hour ahead forecast with 80% prediction interval')
ax.set_ylabel('users')
ax.legend()
plt.show()

### `ask()` — Results mode

Passing `forecast_result` lets the LLM explain the predictions (including the interval columns) and metrics.

In [ ]:
answer = assistant_llm.ask(
    prompt          = (
        "Summarize this 36-hour forecast and explain what the prediction "
        "interval tells me about the uncertainty of the peaks."
    ),
    forecast_result = result_fc,
)
print(answer.explanation)

## 9. Reproducible code

Every executed workflow exposes the **exact standalone script** that produced its results. This is plain skforecast code, deterministic, and ready to move to production. No LLM is involved in generating or running it.

In [ ]:
print(result_fc.code)

## 10. Free-form Q&A

Finally, `ask()` in **Q&A mode** (no data, profile, or result attached) works as a forecasting knowledge assistant, selecting relevant skills automatically.

In [ ]:
answer = assistant_llm.ask(
    prompt = (
        "For hourly demand with strong daily and weekly seasonality, when should "
        "I prefer a direct strategy over a recursive one?"
    ),
)
print(answer.explanation)

## Summary

We built a complete forecasting pipeline for hourly bike-sharing demand:

1. **Profiled** the data and got a recommended forecaster + estimator.
2. **Planned** the detailed configuration, including an 80% prediction interval.
3. **Refined** the plan with domain knowledge about daily/weekly seasonality.
4. **Defined** a time series cross-validation strategy.
5. **Backtested** to evaluate performance out of sample.
6. **Forecasted** the next 36 hours with prediction intervals.
7. Exported the **exact reproducible script**.

Throughout, `ask()` explained each decision and result across its four modes (Explain, Backtest, Results, Q&A), without ever altering the deterministic recommendations.

For a one-line alternative, the fast-path methods `forecast()` and `forecast_code()` run the entire profile -> plan -> execute pipeline in a single call.